# Crosswalk: Anthropic Auto/Aug Task Scores → Occupation-Level Indices

Uses Anthropic's task-level automation vs augmentation breakdown
(from `automation_vs_augmentation_by_task.csv`) + O*NET crosswalk
to produce occupation-level AutoIndex and AugIndex.

Interaction types:
- **Automation** = `directive` + `feedback_loop`
- **Augmentation** = `task_iteration` + `validation` + `learning`
- `filtered` = unclassified (dropped)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Load data
tasks = pd.read_csv('../raw/Anthropic Task/automation_vs_augmentation_by_task.csv')
xwalk = pd.read_csv('../raw/Crosswalks/onet_task_statements.csv')

print(f'Task interaction data: {len(tasks)} tasks')
print(f'O*NET crosswalk: {len(xwalk)} task-occupation pairs')
print(f'Unique occupations: {xwalk["O*NET-SOC Code"].nunique()}')
print()
print('Task columns:', tasks.columns.tolist())
tasks.head()

In [ ]:
# ── Compute auto / aug scores per task ────────────────────────────────────────
# Automation = directive + feedback_loop
# Augmentation = task_iteration + validation + learning
tasks['auto_share'] = tasks['directive'] + tasks['feedback_loop']
tasks['aug_share']  = tasks['task_iteration'] + tasks['validation'] + tasks['learning']
tasks['classified'] = 1 - tasks['filtered']

# Normalize to exclude filtered (so auto + aug = 1 among classified)
tasks['auto_score'] = np.where(tasks['classified'] > 0,
                               tasks['auto_share'] / tasks['classified'], 0)
tasks['aug_score']  = np.where(tasks['classified'] > 0,
                               tasks['aug_share']  / tasks['classified'], 0)

print(f'Tasks with any classified usage: {(tasks["classified"] > 0).sum()}')
print(f'Mean auto_score (among classified): {tasks.loc[tasks["classified"]>0, "auto_score"].mean():.3f}')
print(f'Mean aug_score  (among classified): {tasks.loc[tasks["classified"]>0, "aug_score"].mean():.3f}')
print()
print('Quick check — auto + aug should ≈ 1:')
print((tasks['auto_score'] + tasks['aug_score']).describe())

In [ ]:
# ── Merge task scores onto crosswalk ──────────────────────────────────────────
tasks['task_lower'] = tasks['task_name'].str.lower().str.strip()
xwalk['task_lower'] = xwalk['Task'].str.lower().str.strip()

merged = xwalk.merge(
    tasks[['task_lower', 'auto_score', 'aug_score', 'auto_share', 'aug_share', 'classified']],
    on='task_lower', how='left'
)

n_matched = merged['auto_score'].notna().sum()
print(f'Matched: {n_matched} / {len(merged)} ({100*n_matched/len(merged):.1f}%)')

for col in ['auto_score', 'aug_score', 'auto_share', 'aug_share', 'classified']:
    merged[col] = merged[col].fillna(0.0)

merged.head()

In [ ]:
# ── Also merge penetration scores ────────────────────────────────────────────
pen = pd.read_csv('../raw/Anthropic Task/task_penetration.csv')
pen['task_lower'] = pen['task'].str.lower().str.strip()

merged = merged.merge(pen[['task_lower', 'penetration']], on='task_lower', how='left')
merged['penetration'] = merged['penetration'].fillna(0.0)

print(f'Non-zero penetration: {(merged["penetration"] > 0).sum()} / {len(merged)}')

In [ ]:
# ── Aggregate to occupation level ─────────────────────────────────────────────
merged['weight'] = np.where(merged['Task Type'] == 'Core', 2.0, 1.0)
merged['weight'] *= merged['Incumbents Responding'].fillna(1.0)

def agg_occ(g):
    w = g['weight']
    pen = g['penetration']
    total_w = w.sum()
    return pd.Series({
        'automation_index':    (pen * g['auto_score'] * w).sum() / total_w,
        'augmentation_index':  (pen * g['aug_score']  * w).sum() / total_w,
        'mean_penetration':    (pen * w).sum() / total_w,
        'n_tasks': len(g),
        'n_tasks_classified': (g['classified'] > 0).sum(),
    })

occ_index = merged.groupby(['O*NET-SOC Code', 'Title']).apply(agg_occ).reset_index()

print(f'Occupation-level index: {len(occ_index)} occupations')
print(occ_index[['automation_index', 'augmentation_index', 'mean_penetration']].describe())
print()
print('Top 10 by automation_index:')
print(occ_index.nlargest(10, 'automation_index')[['Title', 'automation_index',
      'augmentation_index', 'mean_penetration']].to_string(index=False))
print()
print('Top 10 by augmentation_index:')
print(occ_index.nlargest(10, 'augmentation_index')[['Title', 'augmentation_index',
      'automation_index', 'mean_penetration']].to_string(index=False))

In [ ]:
# ── Export ────────────────────────────────────────────────────────────────────
occ_index['soc_code'] = occ_index['O*NET-SOC Code'].str.replace('.00', '', regex=False)

export_cols = ['soc_code', 'O*NET-SOC Code', 'Title', 'automation_index',
               'augmentation_index', 'mean_penetration', 'n_tasks']

occ_index[export_cols].to_csv('../output/data/anthropic_occ_index.csv', index=False)
print(f'Exported {len(occ_index)} occupations to output/data/anthropic_occ_index.csv')

In [ ]:
# ── Load CPS occupation data from CSV (college %, n_workers) ──────────────────
occ_col = pd.read_csv('../output/data/occ_college_pct.csv')
occ_col.columns = ['soc_code', 'pct_college', 'n_workers']
occ_col['soc_code'] = occ_col['soc_code'].str.strip()

occ_index['soc_code'] = occ_index['O*NET-SOC Code'].str.replace('.00', '', regex=False)

# Flag computer & mathematical occupations (SOC major group 15)
occ_index['is_compmatch'] = occ_index['O*NET-SOC Code'].str.startswith('15-')

# ── Correlation stats (non-zero occs only) ────────────────────────────────────
from scipy import stats
nonzero = occ_index[(occ_index['automation_index'] > 0) | (occ_index['augmentation_index'] > 0)]
slope, intercept, r_value, p_value, se = stats.linregress(
    nonzero['automation_index'], nonzero['augmentation_index'])
r2 = r_value**2
print(f'Correlation (non-zero occs): slope={slope:.2f}, R²={r2:.3f}, N={len(nonzero)}')
print(f'Computer & Mathematical occs: {occ_index["is_compmatch"].sum()}')

x_fit = np.linspace(0, occ_index['automation_index'].max() * 1.1, 100)

# Corner labels helper
med_a = occ_index['automation_index'].median()
med_s = occ_index['augmentation_index'].median()
corners = {
    'high aug, low auto': occ_index[
        (occ_index['augmentation_index'] > med_s) & (occ_index['automation_index'] <= med_a)
    ].nlargest(1, 'augmentation_index'),
    'high auto, low aug': occ_index[
        (occ_index['automation_index'] > med_a) & (occ_index['augmentation_index'] <= med_s)
    ].nlargest(1, 'automation_index'),
    'both high': occ_index[
        (occ_index['automation_index'] > med_a) & (occ_index['augmentation_index'] > med_s)
    ].nlargest(1, 'mean_penetration'),
    'both low': occ_index[
        (occ_index['automation_index'] <= med_a) & (occ_index['augmentation_index'] <= med_s)
    ].nsmallest(1, 'mean_penetration'),
}

def add_corners(ax):
    for _, rows in corners.items():
        for _, row in rows.iterrows():
            label = row['Title'][:33] + '…' if len(row['Title']) > 35 else row['Title']
            ax.annotate(label,
                        xy=(row['automation_index'], row['augmentation_index']),
                        xytext=(8, 5), textcoords='offset points',
                        fontsize=7, fontstyle='italic',
                        arrowprops=dict(arrowstyle='->', color='black', lw=0.7))

other = occ_index[~occ_index['is_compmatch']]
cm    = occ_index[occ_index['is_compmatch']]


# ── Version 1: plain scatter with fitted line + R² ────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(other['automation_index'], other['augmentation_index'],
           s=20, alpha=0.4, color='#2166ac', edgecolors='none', label='Other')
ax.scatter(cm['automation_index'], cm['augmentation_index'],
           s=30, alpha=0.8, color='#d73027', edgecolors='white', linewidths=0.5,
           label='Computer & Mathematical', zorder=5)
ax.plot(x_fit, intercept + slope * x_fit, color='#555555', lw=1.5, ls='--')
ax.set_xlabel('AutoIndex (directive + feedback loop)')
ax.set_ylabel('AugIndex (task iteration + validation + learning)')
ax.set_title('Occupation-Level: AutoIndex vs AugIndex')
ax.annotate(f'slope = {slope:.2f}, R² = {r2:.3f}, N = {len(nonzero)}',
            xy=(0.98, 0.02), xycoords='axes fraction', ha='right', fontsize=9, color='#555555')
ax.legend(fontsize=9, framealpha=0, loc='upper left')
add_corners(ax)
plt.tight_layout()
plt.savefig('../output/graphs/descriptives/auto_vs_aug_scatter.pdf', bbox_inches='tight')
plt.show()


# ── Version 2: bubble = CPS sample share (from CSV) ──────────────────────────
plot_df = occ_index.merge(occ_col[['soc_code', 'n_workers']], on='soc_code', how='left')
plot_df['n_workers'] = plot_df['n_workers'].fillna(0)
plot_df['share'] = plot_df['n_workers'] / plot_df['n_workers'].sum()
plot_df['bubble_size'] = plot_df['share'] / plot_df['share'].max() * 500

other_b = plot_df[~plot_df['is_compmatch']]
cm_b    = plot_df[plot_df['is_compmatch']]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(other_b['automation_index'], other_b['augmentation_index'],
           s=other_b['bubble_size'], alpha=0.3, color='#2166ac',
           edgecolors='#2166ac', linewidths=0.3, label='Other')
ax.scatter(cm_b['automation_index'], cm_b['augmentation_index'],
           s=cm_b['bubble_size'], alpha=0.7, color='#d73027',
           edgecolors='#d73027', linewidths=0.3,
           label='Computer & Mathematical', zorder=5)
ax.plot(x_fit, intercept + slope * x_fit, color='#555555', lw=1.5, ls='--')
ax.set_xlabel('AutoIndex (directive + feedback loop)')
ax.set_ylabel('AugIndex (task iteration + validation + learning)')
ax.set_title('AutoIndex vs AugIndex (bubble = CPS sample share)')
ax.annotate(f'slope = {slope:.2f}, R² = {r2:.3f}',
            xy=(0.98, 0.02), xycoords='axes fraction', ha='right', fontsize=9, color='#555555')
ax.legend(fontsize=9, framealpha=0, loc='upper left')
add_corners(ax)
plt.tight_layout()
plt.savefig('../output/graphs/descriptives/auto_vs_aug_bubble.pdf', bbox_inches='tight')
plt.show()

print(f'\nR² = {r2:.3f} — highly correlated. Horse-race regression cannot cleanly separate them.')

In [ ]:
# ── Version 3: soc_gr level, Q1-Q4 college %, bubble = sample ─────────────────

occ_with_cps = occ_index.merge(occ_col, on='soc_code', how='inner')
occ_with_cps = occ_with_cps[occ_with_cps['n_workers'] >= 40].copy()
occ_with_cps['soc_major'] = occ_with_cps['O*NET-SOC Code'].str[:2]

print(f'Occupations with 40+ workers: {len(occ_with_cps)}')

def wavg(g, col):
    return np.average(g[col], weights=g['n_workers'])

gr = occ_with_cps.groupby('soc_major').apply(
    lambda g: pd.Series({
        'automation_index': wavg(g, 'automation_index'),
        'augmentation_index': wavg(g, 'augmentation_index'),
        'pct_college': wavg(g, 'pct_college'),
        'n_workers': g['n_workers'].sum(),
    })
).reset_index()

gr['share'] = gr['n_workers'] / gr['n_workers'].sum()
gr['bubble'] = gr['share'] / gr['share'].max() * 600

gr = gr.sort_values('pct_college')
gr['col_q'] = pd.qcut(gr['pct_college'], 4,
                       labels=['Q1 (low college)', 'Q2', 'Q3', 'Q4 (high college)'],
                       duplicates='drop')

q_colors = {'Q1 (low college)': '#2166ac', 'Q2': '#67a9cf',
            'Q3': '#ef8a62', 'Q4 (high college)': '#b2182b'}

fig, ax = plt.subplots(figsize=(9, 7))

for q, col in q_colors.items():
    sub = gr[gr['col_q'] == q]
    ax.scatter(sub['automation_index'], sub['augmentation_index'],
               s=sub['bubble'], alpha=0.6, color=col,
               edgecolors=col, linewidths=0.5, label=f'{q} ({len(sub)})')

for _, row in gr.iterrows():
    ax.annotate(f'SOC {row["soc_major"]}',
                xy=(row['automation_index'], row['augmentation_index']),
                xytext=(5, 3), textcoords='offset points',
                fontsize=6, alpha=0.8)

ax.set_xlabel('AutoIndex (worker-weighted)')
ax.set_ylabel('AugIndex (worker-weighted)')
ax.set_title('Major Occupation Groups: Auto vs Aug (color = college quartile, size = sample)')
ax.legend(fontsize=9, framealpha=0, loc='upper left', title='College %', title_fontsize=9)
plt.tight_layout()
plt.savefig('../output/graphs/descriptives/auto_vs_aug_college.pdf', bbox_inches='tight')
plt.show()
print(f'{len(gr)} major groups')

In [ ]:
# ── Version 4: detailed occ, above/below median college %, CPS only ───────────

det = occ_with_cps.copy()
det = det[det['n_workers'] > 0].copy()

med_col = det['pct_college'].median()
det['col_group'] = np.where(det['pct_college'] >= med_col, 'Above median', 'Below median')

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(det.loc[det['col_group']=='Below median', 'automation_index'],
           det.loc[det['col_group']=='Below median', 'augmentation_index'],
           s=20, alpha=0.5, color='#2166ac', edgecolors='none',
           label=f'Below median ({(det["col_group"]=="Below median").sum()})')

ax.scatter(det.loc[det['col_group']=='Above median', 'automation_index'],
           det.loc[det['col_group']=='Above median', 'augmentation_index'],
           s=20, alpha=0.5, color='#d73027', edgecolors='none',
           label=f'Above median ({(det["col_group"]=="Above median").sum()})')

ax.set_xlabel('AutoIndex (directive + feedback loop)')
ax.set_ylabel('AugIndex (task iteration + validation + learning)')
ax.set_title(f'Detailed Occupations: Auto vs Aug (median college = {med_col:.1f}%)')
ax.legend(fontsize=9, framealpha=0, loc='upper left', title='College %', title_fontsize=9)
plt.tight_layout()
plt.savefig('../output/graphs/descriptives/auto_vs_aug_college_detailed.pdf', bbox_inches='tight')
plt.show()

print(f'{len(det)} occupations (40+ workers)')
print(f'Median college %: {med_col:.1f}%')